# Sentinel Fine-Tuning
Fine-tune Mistral 7B on GoEmotions (emotion classification) + journal data (empathetic analysis)

**Runtime**: T4 GPU (free) — ~45 min per 10k examples, ~4.5 hrs for full 59k

In [ ]:
# Upload the combined dataset (13.2 MB)
from google.colab import files
uploaded = files.upload()
dataset_file = list(uploaded.keys())[0]
print(f"Uploaded: {dataset_file}")

In [ ]:
# Install Unsloth + deps
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install xformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048

In [ ]:
# Load Mistral with 4-bit QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {model.num_parameters(only_trainable=True):,}")

In [ ]:
# Load dataset
dataset = load_dataset("json", data_files=dataset_file, split="train")
print(f"Loaded {len(dataset)} examples")

In [ ]:
# Format with chat template
def format_chat(example):
    messages = [
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example['output']},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

formatted = dataset.map(format_chat)
split = formatted.train_test_split(test_size=0.05, seed=42)
train_data, eval_data = split["train"], split["test"]
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

In [ ]:
args = TrainingArguments(
    output_dir="/content/sentinel-out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=args,
)

In [ ]:
# Start training
trainer_stats = trainer.train()
print(f"Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Save LoRA adapters
model.save_pretrained("/content/sentinel-lora")
tokenizer.save_pretrained("/content/sentinel-lora")
print("LoRA saved to /content/sentinel-lora")
!ls -lh /content/sentinel-lora/

In [ ]:
# Quick test
FastLanguageModel.for_inference(model)

tests = [
    "Analyze the emotion in this text. Respond with the emotion labels that apply.\n\nI can't believe I got the job! I'm over the moon!",
    "Read this journal entry and write an empathetic, human-sounding analysis.\n\nWork was stressful today. My manager criticized my report. Feeling anxious and overwhelmed.",
]

for prompt in tests:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(inputs, max_new_tokens=128, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()
    print(f"\n{'='*60}")
    print(f"Input: {prompt[:60]}...")
    print(f"Response: {response}")

In [ ]:
# Download LoRA (50 MB)
!zip -r /content/sentinel-lora.zip /content/sentinel-lora
from google.colab import files
files.download("/content/sentinel-lora.zip")

## After Download: Apply to Ollama

1. Unzip `sentinel-lora.zip` on your machine
2. Copy to `C:\Users\dhans\Desktop\sentinel3\scripts\sentinel-lora`
3. Update `scripts\Modelfile`:
```
FROM mistral:latest
ADAPTER /path/to/sentinel-lora
SYSTEM """...your existing system prompt..."""
```
4. Run: `ollama create sentinel -f scripts\Modelfile`